In [19]:
import os


def escape_string(text):
    result = []
    for c in text:
        if c == '"':
            result.append('\\"')
        elif c == "\\":
            result.append("\\\\")
        elif c == "\n":
            result.append("\\n")
        elif c == "\r":
            result.append("\\r")
        elif c == "\t":
            result.append("\\t")
        else:
            result.append(c)
    return "".join(result)


def create_db(paths, output_json_path):
    parallelizable_cnt = 0

    with open(output_json_path, "w") as f:
        for path in paths:
            parallelizable = False
            with open(os.path.join(path, "code.c"), "r") as code_f:
                code_content = code_f.read()

            if os.path.exists(os.path.join(path, "pragma.c")):
                parallelizable = True
                parallelizable_cnt += 1
                with open(os.path.join(path, "pragma.c"), "r") as pragma_f:
                    pragma = pragma_f.read()
            else:
                pragma = ""

            json_entry = (
                "{"
                f'"code": "{escape_string(code_content)}", '
                f'"label": "{parallelizable}", '
                f'"pragma": "{escape_string(pragma)}"'
                "}\n"
            )
            f.write(json_entry)

    return {
        "parallelizable": parallelizable_cnt,
        "non_parallelizable": len(paths) - parallelizable_cnt,
        "total": len(paths),
    }

In [20]:
paths = [
    os.path.join(
        "/home/mohamed-ashraf/Desktop/projects/accelera/DB_TEST/database/",
        directory,
    )
    for directory in os.listdir(
        "/home/mohamed-ashraf/Desktop/projects/accelera/DB_TEST/database/"
    )
]

output_json_path = "test.json"

output = create_db(paths, output_json_path)
print(output)

{'parallelizable': 154, 'non_parallelizable': 319, 'total': 473}


In [ ]:
import json

jsonl_path = "output.json"

with open(jsonl_path, "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        print("Code:")
        print(obj["code"])
        print("Label:", obj["label"])
        print("Pragma:", obj["pragma"])
        print("-" * 40)

Code:
for (size_t y = 1; y < (h - 1); ++y)
{
  #pragma acc loop worker independent
  for (size_t x = 1; x < (w - 1); ++x)
  {
    double w_local = c[(y * w) + x];
    double restw = 1.0 - w_local;
    dst[(y * w) + x] = ((w_local * src[(y * w) + x]) + ((((src[((y + 1) * w) + x] + src[((y - 1) * w) + x]) + src[((y * w) + x) + 1]) + src[((y * w) + x) - 1]) * (restw * c_cdir))) + ((((src[(((y - 1) * w) + x) - 1] + src[(((y - 1) * w) + x) + 1]) + src[(((y + 1) * w) + x) - 1]) + src[(((y + 1) * w) + x) + 1]) * (restw * c_cdiag));
  }

  dst[((y * w) + w) - 1] = dst[(y * w) + 1];
  dst[(y * w) + 0] = dst[((y * w) + w) - 2];
}

Label: True
Pragma: omp parallel for
----------------------------------------
Code:
for (size_t y = 0; y < h; ++y)
{
  g1[((y * w) + w) - 1] = (g2[((y * w) + w) - 1] = g1[(y * w) + 1]);
  g1[(y * w) + 0] = (g2[(y * w) + 0] = g1[((y * w) + w) - 2]);
}

Label: False
Pragma: 
----------------------------------------
